<a href="https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
## Method Choice

'''I selected a Random Forest model for this lane.

The objective is to identify content that may represent a search opportunity using historical search performance signals.

Random Forest is suitable because it can capture non-linear relationships between impressions, clicks, position, and engagement metrics without requiring strong distribution assumptions.

The model is intended as a decision-support tool that prioritizes opportunities rather than making final decisions automatically.'''

'I selected a Random Forest model for this lane.\n\nThe objective is to identify content that may represent a search opportunity using historical search performance signals.\n\nRandom Forest is suitable because it can capture non-linear relationships between impressions, clicks, position, and engagement metrics without requiring strong distribution assumptions.\n\nThe model is intended as a decision-support tool that prioritizes opportunities rather than making final decisions automatically.'

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
## Split Design

'''A time-aware split is used.

Earlier observations are used for training and later observations are used for evaluation.

This approach is more realistic because it mirrors how a model would be used in practice, where future data is unavailable at training time.

The split helps reduce leakage and provides a more honest estimate of performance.'''

'A time-aware split is used.\n\nEarlier observations are used for training and later observations are used for evaluation.\n\nThis approach is more realistic because it mirrors how a model would be used in practice, where future data is unavailable at training time.\n\nThe split helps reduce leakage and provides a more honest estimate of performance.'

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
import os

print(os.getcwd())

/content


In [ ]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.56 MiB/s, done.
Resolving deltas: 100% (153/153), done.


In [ ]:
%cd flyrank-ml-internship-starter

/content/flyrank-ml-internship-starter


### Load Data

Load the `queue.csv` file into a pandas DataFrame named `queue`.

In [ ]:
import pandas as pd
import os

# List contents of the current directory to debug file path
print("Contents of current directory:")
!ls -F

# Check if data directory exists and list its contents
if os.path.exists('data'):
    print("\nContents of data directory:")
    !ls -F data/
    if os.path.exists('data/raw'):
        print("\nContents of data/raw directory:")
        !ls -F data/raw/
else:
    print("\n'data' directory not found.")

# Load the queue data from the data directory, correcting the path to the actual file found
queue = pd.read_csv('data/raw/content_refresh_anonymized.csv')

Contents of current directory:
AGENTS.md  DATA_USE.md	LICENSE     README.md	      SETUP.md	   work/
CLAUDE.md  docs/	notebooks/  requirements.txt  skills/
data/	   GUIDE.md	outputs/    scripts/	      submission/

Contents of data directory:
raw/

Contents of data/raw directory:
content_refresh_anonymized.csv


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd

# Print available columns to debug KeyError
print("Available columns in 'queue' DataFrame:")
print(queue.columns.tolist())

features = [
    "impressions_90d",
    "clicks_90d",
    "avg_position"
 ]
data = queue.copy()

# Calculate the 'score' column based on the baseline rule
data['score'] = data['impressions_90d'] * (1 - data['ctr'])

data = data.dropna(subset=features + ['score'])

X = data[features]

# simple proxy target
y = data["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)

print("Model MAE:", mae)

Available columns in 'queue' DataFrame:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Model MAE: 140.47241350000002


| Method | Purpose |
|----------|----------|
| Baseline Rule | Impressions × (1 − CTR) |
| Random Forest | Learns patterns automatically |

The model is evaluated against the same data and target proxy used in the baseline.

The objective is to determine whether learned patterns improve prioritization beyond the simple rule.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
## Error Analysis

'''The model performs best on pages with stable search performance patterns.

Some errors appear on pages with unusual click behavior or highly variable impressions.

The model appears to rely heavily on impressions, clicks, and average position.

Potential sources of error include:

- seasonal search behaviour
- changes in search demand
- measurement noise
- incomplete GA4 availability

The results should be interpreted as directional evidence rather than certainty.'''

'The model performs best on pages with stable search performance patterns.\n\nSome errors appear on pages with unusual click behavior or highly variable impressions.\n\nThe model appears to rely heavily on impressions, clicks, and average position.\n\nPotential sources of error include:\n\n- seasonal search behaviour\n- changes in search demand\n- measurement noise\n- incomplete GA4 availability\n\nThe results should be interpreted as directional evidence rather than certainty.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.